# Decoder-Only Transformer

A minimal GPT-style decoder trained on two sentences. Everything is tiny on purpose:
`vocab_size = 5`, `d_model = 4`, one attention head, one decoder block.

## Architecture

```mermaid
flowchart TD
    X["token ids<br/>(batch=2, seq=5)"] --> EMB["nn.Embedding(5, 4)<br/>(2, 5, 4)"]
    EMB --> POS["PositionEncoding<br/>x + pe[:seq_len]<br/>(2, 5, 4)"]

    subgraph BLOCK["DecoderBlock"]
        direction TB
        ATT["Masked Self-Attention<br/>(2, 5, 4)"]
        N1["LayerNorm(x + attn)<br/>(2, 5, 4)"]
        FF["Feed-Forward<br/>Linear 4-&gt;8, ReLU, Linear 8-&gt;4"]
        N2["LayerNorm(x + ff)<br/>(2, 5, 4)"]
        ATT --> N1 --> FF --> N2
    end

    POS --> ATT
    POS -. residual .-> N1
    N1 -. residual .-> N2

    N2 --> FC["nn.Linear(4, 5)<br/>logits (2, 5, 5)"]
    FC --> LOSS["CrossEntropyLoss<br/>vs targets Y (2, 5)"]
```

## Shapes at every step

| Stage | Tensor | Shape |
|---|---|---|
| input | `X` | `(2, 5)` |
| after embedding | word vectors | `(2, 5, 4)` |
| after position encoding | positioned vectors | `(2, 5, 4)` |
| attention scores | `Q @ K.T / sqrt(4)` | `(2, 5, 5)` — seq x seq |
| after decoder block | contextualized vectors | `(2, 5, 4)` |
| after `fc` | logits | `(2, 5, 5)` — seq x vocab |

> The last two dims are both 5 only by coincidence — `seq_len` happens to equal
> `vocab_size` in this toy setup. The middle 5 is *position*, the trailing 5 is *vocabulary*.




In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [13]:
# =====================================================
# Vocabulary
# =====================================================
token_to_id = {
    "<EOS>": 0,
    "<PAD>": 1,
    "x": 2,
    "y": 3,
    "xy": 4,
    "yx": 5,
    "wife": 6,
    "hubby": 7,
    "daughter": 8, 
    "son": 9,
    "is": 10,   
    "and": 11,   

}

id_to_token = {v: k for k, v in token_to_id.items()}

vocab_size = len(token_to_id)
d_model = 4
max_len = 12

In [14]:
# =====================================================
# Training Sentences
#
# The first <EOS> separates the prompt from the answer.
# The trailing <EOS> is the real end of the sequence -- without
# it, "awesome" is never seen as an input, so the model has no
# idea what follows it and generation can't learn to stop.
# =====================================================
sentences = [
    ["y", "is", "x", "<EOS>", "wife", "<EOS>", "<PAD>", "<PAD>"],
    ["x", "is", "y", "<EOS>", "hubby", "<EOS>", "<PAD>", "<PAD>"],
    ["xy", "is", "<EOS>", "x", "and", "y", "son", "<EOS>"],
    ["yx", "is", "<EOS>", "x", "and", "y", "daughter", "<EOS>"],
]

In [15]:
# =====================================================
# Build Training Data
# Input  = sentence[:-1]
# Target = sentence[1:]
# =====================================================
X = []
Y = []

for sentence in sentences:
    ids = [token_to_id[word] for word in sentence]
    X.append(ids[:-1])
    Y.append(ids[1:])

X = torch.tensor(X)
Y = torch.tensor(Y)

print("Input")
print(X)

print("\nTarget")
print(Y)

Input
tensor([[ 3, 10,  2,  0,  6,  0,  1],
        [ 2, 10,  3,  0,  7,  0,  1],
        [ 4, 10,  0,  2, 11,  3,  9],
        [ 5, 10,  0,  2, 11,  3,  8]])

Target
tensor([[10,  2,  0,  6,  0,  1,  1],
        [10,  3,  0,  7,  0,  1,  1],
        [10,  0,  2, 11,  3,  9,  0],
        [10,  0,  2, 11,  3,  8,  0]])


In [16]:
# =====================================================
# Positional Encoding
# =====================================================
class PositionEncoding(nn.Module):

    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1)

        embedding_index = torch.arange(0, d_model, 2)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

       
        self.register_buffer("pe", pe)

    def forward(self, word_embeddings):
        seq_len = word_embeddings.size(-2)
        return word_embeddings + self.pe[:seq_len]


In [17]:
# =====================================================
# Single Head Masked Attention
# =====================================================
class Attention(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(d_model)

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool()

        scores = scores.masked_fill(mask, -1e9)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        return output

In [18]:
# =====================================================
# Decoder Block
# =====================================================
class DecoderBlock(nn.Module):

    def __init__(self, d_model):
        super().__init__()

        self.attention = Attention(d_model)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 8),
            nn.ReLU(),
            nn.Linear(8, d_model),
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn = self.attention(x)

        x = self.norm1(x + attn)

        ff = self.ff(x)

        x = self.norm2(x + ff)

        return x

In [19]:

# =====================================================
# Decoder Only Transformer
# =====================================================
class DecoderOnlyTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = PositionEncoding(d_model, max_len)
        self.decoder = DecoderBlock(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.position(x)
        x = self.decoder(x)
        logits = self.fc(x)
        return logits



In [20]:

# =====================================================
# Create Model
# =====================================================
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    max_len=max_len,
)

criterion = nn.CrossEntropyLoss(ignore_index=token_to_id["<PAD>"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
)


In [21]:
# =====================================================
# Training
# =====================================================
epochs = 1000

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    loss = criterion(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1),
    )

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  Loss = {loss.item():.4f}")


Epoch    0  Loss = 2.8929
Epoch  100  Loss = 0.3144
Epoch  200  Loss = 0.0542
Epoch  300  Loss = 0.0097
Epoch  400  Loss = 0.0046
Epoch  500  Loss = 0.0028
Epoch  600  Loss = 0.0019
Epoch  700  Loss = 0.0014
Epoch  800  Loss = 0.0011
Epoch  900  Loss = 0.0008


In [22]:
# =====================================================
# Predictions
# =====================================================
print("\nPredictions")

model.eval()

with torch.no_grad():

    logits = model(X)

    predictions = logits.argmax(dim=-1)

print(predictions)

print("\nDecoded Predictions")

for sentence in predictions:

    words = [id_to_token[token.item()] for token in sentence]

    print(words)


Predictions
tensor([[10,  2,  0,  6,  0,  2,  0],
        [10,  3,  0,  7,  0,  9,  0],
        [10,  0,  2, 11,  3,  9,  0],
        [10,  0,  2, 11,  3,  8,  0]])

Decoded Predictions
['is', 'x', '<EOS>', 'wife', '<EOS>', 'x', '<EOS>']
['is', 'y', '<EOS>', 'hubby', '<EOS>', 'son', '<EOS>']
['is', '<EOS>', 'x', 'and', 'y', 'son', '<EOS>']
['is', '<EOS>', 'x', 'and', 'y', 'daughter', '<EOS>']


In [26]:
# =====================================================
# Autoregressive Generation
#
# <EOS> here is the prompt/response separator, not the end
# of the sequence -- the answer comes after it. So the first
# <EOS> is passed over and we stop on the second one.
# =====================================================
print("\nGeneration")

prompt = ["yx"]

generated = prompt.copy()

seen_eos = False

for _ in range(max_len - len(generated)):

    ids = [token_to_id[word] for word in generated]

    x = torch.tensor([ids])

    with torch.no_grad():
        logits = model(x)

    next_token = logits[0, -1].argmax().item()

    next_word = id_to_token[next_token]

    generated.append(next_word)

    if next_word == "<EOS>":

        if seen_eos:
            break

        seen_eos = True

output = " ".join(word for word in generated if word != "<EOS>")
print("Prompt :", prompt)
print("Output :", output)


Generation
Prompt : ['yx']
Output : yx is x and y daughter
